In [1]:
import pandas as pd

rentals = pd.read_csv("data/bike_rentals.csv").copy()

print("Shape:", rentals.shape)
print(rentals.dtypes)
print("price dtype:", rentals["price"].dtype)

empty_columns = rentals.columns[rentals.isna().all()].tolist()
print("Columns with all values missing:", empty_columns)

rentals = rentals.drop(columns=empty_columns)

print("Columns remaining:", rentals.shape[1])
print("Empty columns removed:", len(empty_columns))

Shape: (403, 15)
rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object
price dtype: str
Columns with all values missing: ['coupon_code', 'damage_notes', 'late_fee_note']
Columns remaining: 12
Empty columns removed: 3


In [3]:
print(rentals["price"].head())

# Remove the currency symbol, then the thousands separator
rentals["price"] = rentals["price"].str.replace("SEK ", "", regex=False)
print(rentals["price"].head(3))

rentals["price"] = rentals["price"].str.replace(",", "", regex=False)
print(rentals["price"].head(3))

rentals["price"] = rentals["price"].astype(float)
print("Price dtype:", rentals["price"].dtype)
print(rentals["price"].describe())

rentals.nlargest(3, "price")[["rental_id", "bike_type", "status", "price"]]

0     SEK 855.50
1    SEK 1311.50
2     SEK 350.00
3     SEK 569.50
4     SEK 828.00
Name: price, dtype: str
0     855.50
1    1311.50
2     350.00
Name: price, dtype: str
0     855.50
1    1311.50
2     350.00
Name: price, dtype: str
Price dtype: float64
count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64


,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


In [4]:
price_missing = rentals["price"].isna()
cancelled = rentals["status"].str.casefold().eq("cancelled")

print("Missing prices:", price_missing.sum())
print("Missing price percent:", (price_missing.mean() * 100).round(2), "%")

print(pd.crosstab(
    price_missing.rename("price is missing"),
    cancelled.rename("rental was cancelled"),
))

Missing prices: 51
Missing price percent: 12.66 %
rental was cancelled  False  True 
price is missing                  
False                   352      0
True                      4     47


In [5]:
tf_columns = [
    col for col in rentals.columns
    if rentals[col].notna().any()
    and set(rentals[col].dropna().unique()) <= {"t", "f"}
]
print(tf_columns)

for col in tf_columns:
    rentals[col] = rentals[col].map({"t": True, "f": False})

print(rentals[tf_columns].isna().sum())

['is_member', 'insured', 'paid_online']
is_member      0
insured        3
paid_online    0
dtype: int64


In [6]:
member_col = next(col for col in tf_columns if "member" in col.casefold())

rentals.groupby(member_col).agg(
    rentals_with_price=("price", "count"),
    median_price=("price", "median"),
)

,rentals_with_price,median_price
is_member,,
False,248,756.25
True,104,558.00


In [7]:
# Count missing dates before conversion
print("Missing start_date before:", rentals["start_date"].isna().sum())
print("Missing end_date before:", rentals["end_date"].isna().sum())

# Convert text to dates; unparseable values become NaT
rentals["start_date"] = pd.to_datetime(rentals["start_date"], errors="coerce")
rentals["end_date"] = pd.to_datetime(rentals["end_date"], errors="coerce")

print("Missing start_date after:", rentals["start_date"].isna().sum())
print("Missing end_date after:", rentals["end_date"].isna().sum())
print(rentals[["start_date", "end_date"]].dtypes)

rentals["rental_days"] = (
    rentals["end_date"] - rentals["start_date"]
).dt.days

print(rentals["rental_days"].describe())

# The minimum is negative, which suggests some end dates occur before their start dates.

Missing start_date before: 0
Missing end_date before: 0
Missing start_date after: 2
Missing end_date after: 0
start_date    datetime64[us]
end_date      datetime64[us]
dtype: object
count    401.000000
mean       2.955112
std        1.835206
min       -2.000000
25%        1.000000
50%        3.000000
75%        4.000000
max        7.000000
Name: rental_days, dtype: float64


In [8]:
rating_missing = rentals["rating"].isna()

print("Missing ratings:", rating_missing.sum())
print(pd.crosstab(
    rating_missing.rename("rating is missing"),
    rentals["status"],
))

rentals["has_rating"] = rentals["rating"].notna()
print(rentals["has_rating"].value_counts())

print(rentals["rating"].describe())
print("Median rating:", rentals["rating"].median())

Missing ratings: 115
status             Cancelled  Completed  No-show
rating is missing                               
False                      0        288        0
True                      47         46       22
has_rating
True     288
False    115
Name: count, dtype: int64
count    288.000000
mean       4.225694
std        1.019547
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64
Median rating: 5.0


In [9]:
rentals["child_helmet"] = rentals["helmets_text"].str.contains(
    r"\bchild\b", case=False, na=False
)

rentals["helmets_count"] = pd.to_numeric(
    rentals["helmets_text"].str.extract(r"(\d+)", expand=False),
    errors="coerce",
)

print("Missing helmet counts before filling:", rentals["helmets_count"].isna().sum())

rentals["helmets_count"] = rentals["helmets_count"].fillna(0).astype(int)
print(rentals["helmets_count"].value_counts())

Missing helmet counts before filling: 63
helmets_count
1    193
2    127
0     63
3     20
Name: count, dtype: int64


In [10]:
text_cols = ["station", "bike_type", "status"]

for col in text_cols:
    normalized = rentals[col].str.strip().str.lower()
    print(
        f"{col}: {rentals[col].nunique()} original values, "
        f"{normalized.nunique()} after strip/lower"
    )

# Standardize whitespace and capitalization
for col in text_cols:
    rentals[col] = rentals[col].str.strip().str.title()

print("Rentals per station:")
print(rentals["station"].value_counts())

print("Complete duplicate rows:", rentals.duplicated().sum())
print("Duplicate rental_id values:", rentals["rental_id"].duplicated().sum())

rentals = rentals.drop_duplicates()
rentals = rentals.drop_duplicates(subset="rental_id")

print("Final shape:", rentals.shape)

station: 20 original values, 5 after strip/lower
bike_type: 3 original values, 3 after strip/lower
status: 3 original values, 3 after strip/lower
Rentals per station:
station
Central        107
Södermalm       95
Kungsholmen     89
Östermalm       57
Hammarby        55
Name: count, dtype: int64
Complete duplicate rows: 3
Duplicate rental_id values: 3
Final shape: (400, 16)
